In [ ]:
from bs4 import BeautifulSoup
import requests

#Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def fetch_website_content(url):
    try:
        response = requests.get(url, headers=headers)
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, "html.parser")
        # print(soup.prettify())
        title = soup.title.string if soup.title else "Title not found"
        if soup.body:
            for nonText in soup.body(['script', 'style', 'img', 'input']):
                nonText.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = "Body not found"
        return (title + "\n\n" + text)[:2_000]  # Return the title and the first 2000 characters of the text
    except Exception as e:
        print(f"Error fetching the website: {e}")
        return None


# print(fetch_website_content("https://edwarddonner.com"))

In [ ]:
#Checking if ollama is running on local
requests.get("http://localhost:11434").content

In [ ]:
#Defining System and User prompts for the LLM

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [ ]:
from openai import OpenAI

ollama = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")

def messsage_to_llm(website_content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

def summarize_website(url):
    website = fetch_website_content(url)
    if website:
        messages = messsage_to_llm(website)
        response = ollama.chat.completions.create(model="llama3.2:3b", messages=messages)
        return response.choices[0].message.content
    else:
        return "Failed to fetch website content."
    
print(summarize_website("https://edwarddonner.com"))
